# Análise exploratória e scripts computacionais de T_Fold_CrossValidation_KNNRegressor_GridSearch_1.ipynb

- **Repositório:** `analise-de-dados-fundamentos`  
- **Tecnologias:** Pandas, NumPy  
- **Arquivo Original:** `T_Fold_CrossValidation_KNNRegressor_GridSearch_1.ipynb`  

---


Neste exemplo treinamos e avaliamos um modelo preditivo usando **T-fold-CrossValidation**, ou Validação Cruzada com T Pastas

- técnica de aprendizagem usada: KNN regressor
- tarefa supervisionada: regressão - valor diabetes
- métricas de avaliação: coeficiente de determinação R2, mean_absolute_error

Importando os recursos necessários:
- numpy: biblioteca numérica
- sklearn: biblioteca de machine learning, em especial a técnica Regression Tree, as métricas de avaliação e o model_selection que nos permite executar validação cruzada

In [2]:
# Importa bibliotecas necessárias
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn import model_selection
from sklearn.metrics import mean_absolute_error
from sklearn.datasets import load_diabetes


Carregando a base de dados do problema, representada aqui por X e y, onde:
- X: array contendo N instâncias com M atributos (atributos de entrada do problema)
- y: array contendo o valor do atributo alvo de cada instância em X


In [3]:
import pandas as pd
import numpy as np


# loading attribute names
att=pd.read_csv('glass.csv', index_col=-1, nrows=0).columns.tolist()
print(att)


# loading the data
dataset = np.loadtxt('glass.csv', delimiter=",", skiprows=1)
X = dataset[:,0:9]
y = dataset[:,9]
print(X.shape)




['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe']
(214, 9)


Neste ponto definimos a técnica de Machine Learning a ser utilizada e treinamos o modelo usando Validação Cruzada. No exemplo, uma árvore de regressão.

Observer que, diferente do HOLDOUT, não usamos a função *fit()* e sim *model_selection.cross_val_score()*. Esta função irá treinar T modelos. A cada iteração treina usando T-1 pastas e testa com uma delas. Troca a pasta de teste a cada iteração. Ao final fornece como resultado médio + desvio padrão.

OBS: protocolo muito mais robusto que o HOLDOUT, pois cada instância do problema será teste em alguma iteração.

In [12]:
# Treina do modelo com busca dos parâmetros via GridSearch
# Definição dos parâmetros a serem avaliados

from sklearn.model_selection import GridSearchCV, train_test_split

parameters = [
  {'n_neighbors': [1, 2, 3, 4, 5, 6, 7], 'weights': ['uniform', 'distance'], 'p':[1,2,3,4],'algorithm':['auto', 'ball_tree', 'kd_tree', 'brute']},
  ]

folds=5
model = KNeighborsClassifier()

# separando uma parte para base de validação (20%)
X, X_val, y, y_val = train_test_split(X, y, train_size=0.8, random_state=42)

# GridSearch para customizar os parâmetros sobre base de validação
gs = GridSearchCV(model, parameters, scoring = 'r2', cv=folds, n_jobs=-1)
gs.fit(X_val, y_val)

from tabulate import tabulate
import pandas as pd
df=gs.cv_results_
print(tabulate(df, headers='keys', tablefmt='psql'))
print("Melhores parâmetros encontrados: ", gs.best_params_)

# Recuperando os melhores parâmetros
model=gs.best_estimator_

# Usando a validação cruzada com 5 folds neste exemplo.
T=5 # número de pastas ou folds
result = model_selection.cross_val_score(model, X, y, cv=T)


/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


+-----------------+----------------+-------------------+------------------+-------------------+---------------------+-----------+-----------------+-----------------------------------------------------------------------------+---------------------+---------------------+---------------------+---------------------+---------------------+-------------------+------------------+-------------------+
|   mean_fit_time |   std_fit_time |   mean_score_time |   std_score_time | param_algorithm   |   param_n_neighbors |   param_p | param_weights   | params                                                                      |   split0_test_score |   split1_test_score |   split2_test_score |   split3_test_score |   split4_test_score |   mean_test_score |   std_test_score |   rank_test_score |
|-----------------+----------------+-------------------+------------------+-------------------+---------------------+-----------+-----------------+---------------------------------------------------------------

/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


Avaliação do modelo abaixo:

- A variável *result* que criamos anteriormente tem o R2 calculado pelo *model_selection.cross_val_score()*.

- A função *model_selection.cross_val_predict()* retorna o valor predito para cada exemplo de teste.

- A função mean_absolute_error() retorna o erro médio absoluto calculado sobre a base de teste.

In [5]:

# Mostrando R2 médio e desvio padrão calculados na validação cruzada.
print("\nCross Validation Results %d folds:" % T)
print("R2 médio: %.5f" % result.mean())
print("Mean Std: %.5f" % result.std())

# Calculando o valor para cada exemplo de teste
y_pred = model_selection.cross_val_predict(model, X, y, cv=T)

# Exemplo mostrando o resultado previsto para a primeira instância de teste
print("Primeira instância na base de teste apresenta valor diabetes: %d" % y_pred[0])

# Calculando o erro médio absoluto
mae=mean_absolute_error(y, y_pred)
print("Mean Absolute Error (MAE) calculado na base de teste: %.5f" % mae)

# salvando o modelo
from joblib import dump, load
with open("model.mod", 'wb') as fo:
    dump(model, fo)


Cross Validation Results 5 folds:
R2 médio: 0.65496
Mean Std: 0.07528
Primeira instância na base de teste apresenta valor diabetes: 2
Mean Absolute Error (MAE) calculado na base de teste: 0.66667
